# 07. Paoay: boundaries and references

Review the study boundary, reference polygons and original KMZ notes. Record suitability for later comparisons without assigning crop labels or observation dates from calendar notes. Local files only; the model and satellite data remain unchanged.

See the [workflow guide](../../docs/paoay_workflow.md).


## 1. Setup

Use the existing project Python kernel. Put the two KMZ files in `data/paoay/`, or set `paoay_kmz_root` in ignored `settings.local.json`. Existing inputs, editable reviews and older bundles are preserved.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import references, reference_suitability
from crop_pipeline.analysis_workflow import write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

kmz_root = local_path(local_settings.get("paoay_kmz_root", "./data/paoay"))
kmz_paths = [kmz_root / "00 - Region of Interest-2.kmz", kmz_root / "Crop Area.kmz"]
roi_name = local_settings.get("paoay_roi_name", "Region of Interest")
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
reference_root = report_folder.parent / "paoay/references"
show_preview = True
stages = ["Read KMZ files", "Check reference geometry", "Read source notes", "Save reference handoff", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook(
    "07_paoay_reference_review", stages, report_folder.parent / "progress",
    {"inputs": [str(path) for path in kmz_paths], "roi_name": roi_name},
)
print("Input folder:", kmz_root)
print("Reference outputs:", reference_root)

reference_pointer = reference_root / "reference_stage.json"
write_stage_pointer(reference_pointer, status="pending", reason="Notebook 07 has not finished the current reference review")


## 2. Read the KMZ files

Read placemark names, polygon boundaries, description notes and any embedded metadata. The ZIP contents are read in memory; the original KMZs are not edited. Source notes are data, not instructions or confirmed labels.


In [ ]:
pipeline_progress.begin_stage("Read KMZ files")
records, sources = references.read_inputs(kmz_paths)
show_table([{"file": source["name"], "bytes": source["bytes"]} for source in sources], title="Reference input files")
print("Placemarks read:", len(records))
pipeline_progress.end_stage()


## 3. Check boundaries and overlaps

Identify the study boundary by its exact name. Report invalid geometry, areas outside the boundary and overlapping reference areas. Do not repair or merge them automatically. An area polygon is not assumed to be one independently verified field.


In [ ]:
pipeline_progress.begin_stage("Check reference geometry")
inspection = references.inspect_records(records, roi_name=roi_name)
columns = ("name", "role", "area_ha", "geometry_status", "location", "notes_present", "evaluation_status")
show_table([{key: row[key] for key in columns} for row in inspection["audit"]], title="Study boundary and reference areas")
show_table(inspection["overlaps"], title="Overlapping reference areas")
print("Area measurements use:", inspection["metric_crs"])
pipeline_progress.end_stage()


## 4. Read the original notes

Descriptions may contain crop calendars, mixed crops or incomplete dates. Keep those uncertainties. No crop, observation year, planting date or evaluation label is assigned automatically.


In [ ]:
pipeline_progress.begin_stage("Read source notes")
for record in records:
    print("\n" + record["name"])
    print(record["notes"] or "No description notes.")
pipeline_progress.end_stage()


## 5. Save references and review suitability

Keep the immutable geometry/notes snapshot and the existing evidence-review CSV. A separate suitability checklist records area type (single-crop, rotation/mixed, seedlings, non-target crop or unreviewed), temporal compatibility and caveats. Review entries manually; they do not become training labels or independent ground truth. Reruns never overwrite edits.


In [ ]:
pipeline_progress.begin_stage("Save reference handoff")
reference_handoff = references.export_reference_bundle(records, sources, reference_root, roi_name=roi_name)
bundle = references.load_reference_bundle(reference_handoff)
suitability_file, suitability_rows = reference_suitability.prepare_review(reference_handoff)
show_table(suitability_rows, title="Reference suitability — manual review")
print("Editable suitability checklist:", suitability_file)
write_stage_pointer(reference_pointer, reference_handoff)
print("Reference handoff:", reference_handoff)
print("Editable reference review:", bundle["review_file"])
pipeline_progress.end_stage()


## 6. Next step

Review the checklist before interpreting maps. General crop calendars cannot confirm historical occupancy. Notebook 08 prepares both Sentinel-2 and Dynamic World for one explicit period; a historical baseline and a January field-check map must be distinguished.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
manifest = bundle["manifest"]
print("Study boundary:", roi_name)
print("Reference areas:", manifest["area_count"])
print("Invalid area geometries:", manifest["invalid_areas"])
print("Overlapping area pairs:", manifest["overlap_pairs"])
print("Verified crop labels: none assigned by this notebook.")
print("Year and four-month period: not selected.")
if show_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    figure = references.plot_reference_areas(bundle["records"], manifest["roi_id"], manifest["metric_crs"])
    preview = reference_root / "previews" / (manifest["bundle_id"] + ".png")
    preview.parent.mkdir(parents=True, exist_ok=True)
    if not preview.exists():
        figure.savefig(preview, dpi=150)
    display(figure)
    plt.close(figure)
    print("Outline preview:", preview)
print("Next: notebook 08 prepares matching Sentinel-2 and Dynamic World inputs.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
